# Study 1 four-condition comparison, Kaggle T4

Same Tesla T4. Do not switch GPUs. Same Qwen3-4B revision, FP16, greedy decoding.

Uses only `data/plan_dev.jsonl` (12 families, 36 examples). Do **not** load `plan_fresh`, `data/test.jsonl`, `hop_test`, or a `plan_test` file.

Do **not** resume Study 0 folders. Keep `results_stage_plan0`, `_corrected`, `_diag`, and `_e2e`.

This run is the core comparison, not compression and not length-control diagnostics.

- Encode A's local-task prompt as **C0** (no extra latent steps).
- Continue 40 latent steps on an independent copy as **C40**.
- Branch from copies: text from C0, text from C40, full KV from C0, full KV from C40.
- The two text conditions use the **same** intermediate-result handoff, format, and 256-token allowance.
- Also run source-evidence text from C0 (A selects facts; not the hop retriever), B without a message, and mismatched C40 text/KV.
- B always uses the frozen reason-then-answer procedure.

Primary contrasts: $\Delta_{KV}$ = acc(full C40) − acc(full C0); $\Delta_{text}$ = acc(text C40) − acc(text C0). These measure extra rollout, not “reasoning vs no reasoning.”

288 B evaluations. Resume is on for `/kaggle/working/results_stage_study1` only. Download that folder during the run.

Keep every example. These are 12 families with three versions each, not 36 independent scenarios.

In [ ]:
import os, json, shutil, sys
from pathlib import Path

os.environ["LATENT_RELAY_INFER_DTYPE"] = "fp16"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

ON_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()

candidates = [Path.cwd()]
if Path("/kaggle/input").is_dir():
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if d not in {".git", ".pytest_cache"}]
        if "stage_study1.py" in files and Path(root).name == "src":
            candidates.append(Path(root).parent)

REPO = None
for cand in candidates:
    if (cand / "src" / "stage_study1.py").is_file() and (cand / "data" / "plan_dev.jsonl").is_file():
        REPO = cand
        break
if REPO is None:
    raise FileNotFoundError("Need src/stage_study1.py and data/plan_dev.jsonl. Re-upload the repo zip.")

if ON_KAGGLE:
    dest = WORK / "latent-relay-audit"
    if str(REPO).startswith("/kaggle/input"):
        shutil.copytree(REPO, dest, dirs_exist_ok=True)
    elif not (dest / "src" / "stage_study1.py").is_file():
        shutil.copytree(REPO, dest, dirs_exist_ok=True)
    REPO = dest
    os.chdir(REPO)
else:
    os.chdir(REPO)

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("REPO", REPO.resolve())
print("plan_dev", sum(1 for _ in (REPO / "data" / "plan_dev.jsonl").open(encoding="utf-8")))
assert not (REPO / "data" / "plan_test.jsonl").exists()

In [ ]:
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.57.3", "tokenizers==0.22.2", "huggingface_hub==0.36.2",
    "safetensors==0.7.0", "accelerate==1.13.0", "datasets==4.4.1",
])
import torch, transformers
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print("cuda", torch.cuda.is_available(), gpu)
print("transformers", transformers.__version__)
assert transformers.__version__.startswith("4.57")
if gpu is not None and "T4" not in gpu:
    raise RuntimeError(f"Stay on the locked Tesla T4. Got {gpu}.")

In [ ]:
subprocess.check_call([sys.executable, "scripts/fetch_upstream.py"])
print("Dest /kaggle/working/results_stage_study1")
print("Do not resume Study 0 folders. No compressor in this run.")

In [ ]:
rc = subprocess.call([sys.executable, "-m", "src.stage_study1", "--split", "plan_dev"])
print("study1 exit", rc)

In [ ]:
src = WORK / "results_stage_study1"
if not (src / "summary.json").is_file():
    src = REPO / "results" / "stage_study1" / "run"
print("reading", src)
for name in ("summary.json", "manifest.json"):
    path = src / name
    if path.is_file():
        print("====", name)
        print(path.read_text(encoding="utf-8")[:12000])